<a href="https://colab.research.google.com/github/AMJ5670886/thinkpalm-agentai-ajoldmartinjose-reAct_Agent/blob/master/src/minimal_react_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
import re
import json
import requests
from openai import OpenAI
from google.colab import userdata  # Import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["TMDB_API_KEY"] = userdata.get("TMDB_API_KEY")

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

TMDB_KEY = os.environ["TMDB_API_KEY"]
TMDB_BASE = "https://api.themoviedb.org/3"
TMDB_HEADERS = {
    "Authorization": f"Bearer {TMDB_KEY}",  # v4 token
    "accept": "application/json",
}

_YEAR_SUFFIX = re.compile(r"^(?P<title>.+?)\s+(?P<year>(?:19|20)\d{2})$")


def parse_title_and_year(query: str) -> tuple[str, int | None]:
    """If query ends with ' ... 1997', return ('...', 1997); else (stripped query, None)."""
    q = query.strip()
    m = _YEAR_SUFFIX.match(q)
    if not m:
        return q, None
    title = m.group("title").strip()
    year = int(m.group("year"))
    return (title, year) if title else (q, None)


def film_search(film_name: str) -> str:
    try:
        title_q, year_filter = parse_title_and_year(film_name)
        s = requests.get(
            f"{TMDB_BASE}/search/movie",
            headers=TMDB_HEADERS,
            params={
                "query": title_q,
                "include_adult": "false",
                "language": "en-US",
                "region": "IN",
            },
            timeout=10,
        ).json()
        if "status_code" in s and s.get("success") is False:
            return f"TMDB auth/error: {s.get('status_message')}"
        results = s.get("results", [])
        if not results:
            return f"No TMDB results for '{title_q}'."
        results = sorted(results, key=lambda r: r.get("popularity", 0), reverse=True)
        if year_filter is not None:
            results = [
                r
                for r in results
                if (r.get("release_date") or "")[:4] == f"{year_filter:04d}"
            ]
            if not results:
                return f"No TMDB results for '{title_q}' released in {year_filter}."
        results = results[:5]
        blocks = []
        for r in results:
            mid = r["id"]
            details = requests.get(
                f"{TMDB_BASE}/movie/{mid}",
                headers=TMDB_HEADERS,
                params={"append_to_response": "credits", "language": "en-US"},
                timeout=10,
            ).json()
            crew = details.get("credits", {}).get("crew", [])
            cast = details.get("credits", {}).get("cast", [])
            directors = [c["name"] for c in crew if c.get("job") == "Director"] or ["Unknown"]
            top_cast = [c["name"] for c in cast[:5]] or ["Unknown"]
            blocks.append(
                f"TITLE: {details.get('title')} ({details.get('original_title')})\n"
                f"YEAR: {(details.get('release_date') or '????')[:4]}\n"
                f"LANGUAGE: {details.get('original_language')}\n"
                f"DIRECTOR: {', '.join(directors)}\n"
                f"CAST: {', '.join(top_cast)}\n"
                f"OVERVIEW: {details.get('overview') or ''}"
            )
        return "\n\n=====\n\n".join(blocks)
    except Exception as e:
        return f"TMDB error: {e}"


TOOLS = {"film_search": film_search}
TOOL_DESC = """
Available tools:
1) film_search(film_name: str) -> str
   Searches TMDB across all languages and includes latest releases.
   Returns up to 5 matching films with director, year, language, and cast.
   If film_name ends with a space and year (e.g. "Dune 2021"), only that release year is returned.
""".strip()

# ----------------------------
# 2) ReAct prompt
# ----------------------------
SYSTEM_PROMPT = f"""
You are a minimal ReAct agent that answers questions about FILMS in any language,
including the latest releases.

Internally reason step-by-step using:

Thought: ...
Action: film_search OR finish
Action Input: JSON string, e.g. {{\"film_name\": \"Manjummel Boys\"}}
Observation: tool result (provided by system)

The Observation may contain MULTIPLE candidate films separated by "=====".
Treat each block as a separate film.

When ready, output ONLY this final block:

Final Answer:

For EACH film block in the Observation, output:

--- Match <n>
Title: <title>
Director: <director or Unknown>
Year: <year or Unknown>
Main Cast: <comma-separated actors>
Language: <full language name, e.g. Malayalam, Hindi, Tamil, English>

Rules:
- Always call film_search at least once.
- Do NOT merge different films into one block.
- Roman/English script for names.
- No Thought/Action lines in the final reply.
- If the user gives a year with the title (e.g. "Film 2019"), pass that full string in film_name so the tool can filter by year.

{TOOL_DESC}
""".strip()


def call_llm(messages, temperature=0):
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content


def parse_action_block(text: str):
    action_match = re.search(r"Action:\s*(.+)", text)
    action_input_match = re.search(r"Action Input:\s*(.+)", text)
    final_match = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)
    action = action_match.group(1).strip() if action_match else None
    action_input = action_input_match.group(1).strip() if action_input_match else None
    final_answer = final_match.group(1).strip() if final_match else None
    return action, action_input, final_answer


def run_film_agent(film_name: str, max_steps: int = 5) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Find info about the film: {film_name}"},
    ]

    for _ in range(max_steps):
        assistant_text = call_llm(messages)
        messages.append({"role": "assistant", "content": assistant_text})

        action, action_input, final_answer = parse_action_block(assistant_text)

        if final_answer:
            return f"Final Answer:\n{final_answer.strip()}"

        if not action:
            return "Agent error: no Action found."

        if action.lower() == "finish":
            messages.append({"role": "user", "content": "Provide `Final Answer:` now."})
            continue

        if action not in TOOLS:
            return f"Agent error: unknown tool `{action}`."

        try:
            parsed = json.loads(action_input)
            if isinstance(parsed, dict):
                tool_arg = parsed.get("film_name") or parsed.get("query") or str(parsed)
            else:
                tool_arg = str(parsed)
        except Exception:
            tool_arg = (action_input or "").strip('"')

        observation = TOOLS[action](tool_arg)
        messages.append({"role": "user", "content": f"Observation: {observation}"})

    return "Agent stopped: max steps reached."

LANG_MAP = {
    "ml": "Malayalam",
    "ta": "Tamil",
    "hi": "Hindi",
    "te": "Telugu",
    "kn": "Kannada",
    "en": "English",
}


def lookup_film_direct(query: str, top_n: int = 3) -> str:
    title_q, year_filter = parse_title_and_year(query)
    s = requests.get(
        f"{TMDB_BASE}/search/movie",
        headers=TMDB_HEADERS,
        params={
            "query": title_q,
            "include_adult": "false",
            "language": "en-US",
            "region": "IN",
        },
        timeout=10,
    ).json()
    results = s.get("results", []) or []
    results = sorted(results, key=lambda r: r.get("popularity", 0), reverse=True)
    if year_filter is not None:
        results = [
            r
            for r in results
            if (r.get("release_date") or "")[:4] == f"{year_filter:04d}"
        ]
        if not results:
            return f"No TMDB results for '{title_q}' released in {year_filter}."
    results = results[:top_n]
    if not results:
        return f"No TMDB results for '{title_q}'."

    out = []
    for i, r in enumerate(results, 1):
        d = requests.get(
            f"{TMDB_BASE}/movie/{r['id']}",
            headers=TMDB_HEADERS,
            params={"append_to_response": "credits", "language": "en-US"},
            timeout=10,
        ).json()

        crew = (d.get("credits") or {}).get("crew") or []
        cast = (d.get("credits") or {}).get("cast") or []
        directors = [c["name"] for c in crew if c.get("job") == "Director"]
        top_cast = [c["name"] for c in cast[:6]]
        year = (d.get("release_date") or "????")[:4]

        out.append(
            f"--- Match {i}\n"
            f"Title: {d.get('title') or d.get('original_title')}\n"
            f"Director: {', '.join(directors) if directors else 'Unknown'}\n"
            f"Year: {year if year != '????' else 'Unknown'}\n"
            f"Main Cast: {', '.join(top_cast) if top_cast else 'Unknown'}\n"
            f"Language: {LANG_MAP.get(d.get('original_language'), d.get('original_language') or 'Unknown')}"
        )
    return "\n\n".join(out)


def main():
    while True:
        film = input("\nEnter film name (or 'exit'): ").strip()
        if film.lower() == "exit":
            break
        if not film:
            continue
        print(lookup_film_direct(film))


main()


Enter film name (or 'exit'): too fast too furious
No TMDB results for 'too fast too furious'.

Enter film name (or 'exit'): fast and furious 2
--- Match 1
Title: Fast Forever
Director: Louis Leterrier
Year: 2028
Main Cast: Vin Diesel
Language: English

--- Match 2
Title: 2 Fast 2 Furious
Director: John Singleton
Year: 2003
Main Cast: Paul Walker, Tyrese Gibson, Eva Mendes, Cole Hauser, Ludacris, James Remar
Language: English

--- Match 3
Title: The Turbo Charged Prelude for 2 Fast 2 Furious
Director: Philip G. Atwell
Year: 2003
Main Cast: Paul Walker, Minka Kelly, Peter Aylward, Rodney Neil, Vin Diesel
Language: English

Enter film name (or 'exit'): exit
